# Dense structural channels: three-set analysis (iteration 29)

Reads `channels_<set>.csv` from `Coarse_Channels.ipynb` (3 queries per photo, channels edge / dog / texture / orient / fuse_eo / fuse_all, per query: best peak, z, error in sigma, top-5 hit, decoy peak with the true place blanked). Kill rule (written before the run): agreement of the edge answer with a channel answer within 1 sigma must give 0 wrong answers on all three sets, answer at least 40 % of the photos above 800 m on the new set, and show 0 false agreements on the decoys.

In [ ]:
import numpy as np, pandas as pd
R = "../results/coarse_channels/"
def pts(e): return np.where(e<=1,10,np.where(e<=2,6,np.where(e<=4,3,np.where(e<=8,0,-20))))
sets = {"new (332)": "highalt_dev332", "official (first 150)": "official150", "NUDT (first 150)": "nudt150"}
tabs = {n: pd.read_csv(R + "channels_%s.csv" % t) for n, t in sets.items()}
CH = ["edge", "dog", "texture", "orient", "fuse_eo", "fuse_all"]
def best_per_photo(t):
    return t.loc[t.groupby(["photo", "channel"]).z.idxmax()].set_index(["photo", "channel"])
for n, t in tabs.items():
    print("\n==", n, len(t.photo.unique()), "photos")
    b = best_per_photo(t).reset_index(); b["band"] = pd.cut(b.height, [0, 500, 800, 1300, 2200])
    print(b.groupby(["band", "channel"], observed=True).agg(n=("err","size"), right2=("err", lambda e:(e<=2).mean()), wrong=("err", lambda e:(e>8).mean()),
          top5=("top5", lambda x:(x<=2).mean()), pts=("err", lambda e: pts(e.values).mean())).round(2).unstack("channel")[["right2","wrong"]].to_string())

## Agreement gate: the edge answer, kept only when an independent channel's answer lies within 1 sigma of it

In [ ]:
def gate_table(t, label, min_height=0):
    b = best_per_photo(t); photos = sorted(set(i[0] for i in b.index))
    rows = []
    for p in photos:
        e = b.loc[(p, "edge")]
        if e.height < min_height: continue
        agree = {g: float(np.hypot(b.loc[(p, g)].x - e.x, b.loc[(p, g)].y - e.y) / e.sigma_m <= 1.0) for g in ["dog", "texture", "orient"]}
        # decoys: best decoy peak (true place blanked) over the queries of each channel
        d = t[t.photo == p]; de = d[d.channel == "edge"].loc[lambda x: x.dz.idxmax()] if d[d.channel == "edge"].dz.notna().any() else None
        dagree = {}
        for g in ["dog", "texture", "orient"]:
            dg = d[d.channel == g]
            dagree[g] = float(np.hypot(dg.loc[dg.dz.idxmax()].dx - de.dx, dg.loc[dg.dz.idxmax()].dy - de.dy) / e.sigma_m <= 1.0) if (de is not None and dg.dz.notna().any()) else np.nan
        rows.append(dict(photo=p, height=e.height, err=e.err, pts=float(pts(np.array([e.err]))[0]), **{"agree_"+g: v for g, v in agree.items()}, **{"decoy_"+g: v for g, v in dagree.items()}))
    f = pd.DataFrame(rows); f["n_agree"] = f[["agree_dog","agree_texture","agree_orient"]].sum(axis=1); f["any_decoy"] = f[["decoy_dog","decoy_texture","decoy_orient"]].max(axis=1)
    print("\n==", label, "n =", len(f), " edge-only (always answer): pts/photo %.2f wrong %d" % (f.pts.mean(), (f.err > 8).sum()))
    out = []
    for name, mask in [("dog", f.agree_dog == 1), ("texture", f.agree_texture == 1), ("orient", f.agree_orient == 1), ("any 1 of 3", f.n_agree >= 1), ("2 of 3", f.n_agree >= 2)]:
        s = f[mask]; out.append(dict(gate=name, answered=len(s), share=round(len(s)/len(f), 2), wrong=int((s.err > 8).sum()), right2=round((s.err <= 2).mean(), 2) if len(s) else None, pts_per_photo=round(s.pts.sum()/len(f), 2)))
    print(pd.DataFrame(out).to_string(index=False))
    dec = {g: np.nanmean(f["decoy_" + g]) for g in ["dog", "texture", "orient"]}
    print("false agreement on decoys (share of photos where the edge decoy and the channel decoy peaks coincide within 1 sigma):", {g: round(v, 3) for g, v in dec.items()}, " any:", round(np.nanmean(f.any_decoy), 3))
    return f
gates = {}
for n, t in tabs.items():
    gates[n] = gate_table(t, n + ", all heights")
    if n.startswith("new"): gate_table(t, n + ", above 800 m", 800)

## Does fusing improve the SEARCH itself? (answer = best-z query of the fused map)

The tables above already show `fuse_eo` and `fuse_all` next to `edge` per band. Fusion is judged by right2 up and wrong down against `edge` in the same band.